# Filters and missing data

This tutorial covers two everyday data-cleaning steps in MCDA:
**filtering** out alternatives that should never enter the ranking, and
**imputing** criteria values that are simply missing from the data.

## Case

A factory has to choose raw-material suppliers among eight candidates
(*S1* to *S8*). Four criteria are considered:

1. **Price** (USD/unit). Sense of optimality, $Minimize$.
2. **Quality** (audit score, 0-100). Sense of optimality, $Maximize$.
3. **Delivery** (days). Sense of optimality, $Minimize$.
4. **Tier** (internal supplier tier, 1 to 5, 5 being the best). Sense of
   optimality, $Maximize$.

Two suppliers (*S3*, *S7*) have not been quality-audited yet, and two
(*S4*, *S8*) have no confirmed delivery time. Additionally, *S5* belongs
to *Tier 1*, which the purchasing department has blacklisted for this
kind of purchase.

The full (dirty) decision matrix

In [1]:
import numpy as np
import skcriteria as skc

alternatives = ["S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8"]
price = [420, 380, 610, 350, 275, 500, 460, 395]
quality = [88, 91, np.nan, 75, 60, 82, np.nan, 95]
delivery = [12, 9, 20, np.nan, 30, 14, 10, np.nan]
tier = [4, 5, 2, 3, 1, 4, 5, 3]

dm = skc.mkdm(
    matrix=np.column_stack([price, quality, delivery, tier]),
    objectives=[min, max, min, max],
    alternatives=alternatives,
    criteria=["Price", "Quality", "Delivery", "Tier"],
)
dm

,Price[▼ 1.0],Quality[▲ 1.0],Delivery[▼ 1.0],Tier[▲ 1.0]
S1,420.0,88.0,12.0,4.0
S2,380.0,91.0,9.0,5.0
S3,610.0,NaN,20.0,2.0
S4,350.0,75.0,NaN,3.0
S5,275.0,60.0,30.0,1.0
S6,500.0,82.0,14.0,4.0
S7,460.0,NaN,10.0,5.0
S8,395.0,95.0,NaN,3.0


## 1. Recap: generic and arithmetic filters

The [Dominance and satisfaction analysis](sufdom.ipynb) tutorial already
covers `filters.Filter` (arbitrary function per criterion) and the full
arithmetic family `FilterGT`, `FilterGE`, `FilterLT`, `FilterLE`,
`FilterEQ`, `FilterNE` in detail, so we won't repeat that here.

One detail worth remembering: a single filter instance can carry
conditions on **several criteria at once** — the dict keys are the
criteria and an alternative survives only if it satisfies *all* of them.
For example, "keep only suppliers with `Price <= 500` **and**
`Tier >= 3`\" is a single `FilterLE`/`FilterGE` combo:

In [2]:
from skcriteria.preprocessing import filters

cheap_and_reliable = filters.FilterLE({"Price": 500}).transform(dm)
cheap_and_reliable = filters.FilterGE({"Tier": 3}).transform(cheap_and_reliable)
cheap_and_reliable

,Price[▼ 1.0],Quality[▲ 1.0],Delivery[▼ 1.0],Tier[▲ 1.0]
S1,420.0,88.0,12.0,4.0
S2,380.0,91.0,9.0,5.0
S4,350.0,75.0,NaN,3.0
S6,500.0,82.0,14.0,4.0
S7,460.0,NaN,10.0,5.0
S8,395.0,95.0,NaN,3.0


## 2. Set-based filters: `FilterIn` and `FilterNotIn`

Sometimes the condition is not a numeric threshold but membership in a
set of allowed (or forbidden) values. `skcriteria.preprocessing.filters`
provides two set-based filters for this:

- `FilterIn`: keeps alternatives whose criterion value **is** in a given
  collection.
- `FilterNotIn`: keeps alternatives whose criterion value **is not** in a
  given collection.

Our purchasing department blacklists *Tier 1* suppliers outright — this
is a natural fit for `FilterNotIn`.

In [3]:
dmf = filters.FilterNotIn({"Tier": [1]}).transform(dm)
dmf

,Price[▼ 1.0],Quality[▲ 1.0],Delivery[▼ 1.0],Tier[▲ 1.0]
S1,420.0,88.0,12.0,4.0
S2,380.0,91.0,9.0,5.0
S3,610.0,NaN,20.0,2.0
S4,350.0,75.0,NaN,3.0
S6,500.0,82.0,14.0,4.0
S7,460.0,NaN,10.0,5.0
S8,395.0,95.0,NaN,3.0


`S5`, the only *Tier 1* supplier, was removed. `FilterIn` works the same
way but with the opposite logic — for instance
`filters.FilterIn({"Tier": [4, 5]})` would keep only the *preferred*
tiers.

<div class="alert alert-info">

**Note:**

`skcriteria.preprocessing.filters` also implements
`FilterNonDominated`, which removes dominated alternatives instead of
filtering by a criterion value. It is covered in depth, together with the
whole `DecisionMatrix.dominance` accessor, in the
[Dominance and satisfaction analysis](sufdom.ipynb) tutorial — in a real
pipeline you would typically chain a set/arithmetic filter with
`FilterNonDominated`.

</div>

We will keep using `dmf` (blacklist already applied) for the rest of this
tutorial — it still has missing `Quality` and `Delivery` values for *S3*,
*S4*, *S7* and *S8*.

## 3. Missing data: `SimpleImputer`

MCDA methods cannot operate on `NaN` values, so before ranking we need to
fill them in somehow. The simplest strategy is
`skcriteria.preprocessing.impute.SimpleImputer`, a thin wrapper around
[`sklearn.impute.SimpleImputer`](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html),
which replaces every missing value in a criterion using a summary
statistic **of that same criterion** (mean, median, most frequent value,
or a constant).

In [4]:
from skcriteria.preprocessing import impute

mean_imputed = impute.SimpleImputer(strategy="mean").transform(dmf)
median_imputed = impute.SimpleImputer(strategy="median").transform(dmf)

display(mean_imputed)
display(median_imputed)

,Price[▼ 1.0],Quality[▲ 1.0],Delivery[▼ 1.0],Tier[▲ 1.0]
S1,420.0,88.0,12.0,4.0
S2,380.0,91.0,9.0,5.0
S3,610.0,86.2,20.0,2.0
S4,350.0,75.0,13.0,3.0
S6,500.0,82.0,14.0,4.0
S7,460.0,86.2,10.0,5.0
S8,395.0,95.0,13.0,3.0


,Price[▼ 1.0],Quality[▲ 1.0],Delivery[▼ 1.0],Tier[▲ 1.0]
S1,420.0,88.0,12.0,4.0
S2,380.0,91.0,9.0,5.0
S3,610.0,88.0,20.0,2.0
S4,350.0,75.0,12.0,3.0
S6,500.0,82.0,14.0,4.0
S7,460.0,88.0,10.0,5.0
S8,395.0,95.0,12.0,3.0


Both strategies agree on `Price` and `Tier` (no missing values there),
but disagree on the imputed `Quality`/`Delivery`: the mean is pulled by
the whole distribution while the median is more robust to the spread
between suppliers. A `strategy="constant"` with `fill_value=0` is also
available, useful when a missing value should be treated as "worst case"
rather than "typical case".

<div class="alert alert-warning">

**Note:**

`SimpleImputer` looks at each criterion **independently** — it ignores
any relationship between `Quality`, `Delivery`, `Price`, etc. The next
section introduces two imputers that do take those relationships into
account.

</div>

## 4. Multivariate imputation: `IterativeImputer` and `KNNImputer`

- `IterativeImputer` models each criterion with missing values as a
  function of the *other* criteria, in a round-robin fashion, until the
  estimates converge. It is still considered *experimental* in
  scikit-learn, so it must be explicitly enabled.
- `KNNImputer` fills a missing value with the (weighted) average of that
  same criterion among the `n_neighbors` most similar alternatives (in
  terms of the criteria that *are* known).

Both take the whole matrix into account instead of a single column,
which usually gives more realistic estimates when criteria are
correlated.

In [5]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401

iterative_imputed = impute.IterativeImputer(random_state=42).transform(dmf)
knn_imputed = impute.KNNImputer(n_neighbors=2).transform(dmf)

display(iterative_imputed)
display(knn_imputed)

,Price[▼ 1.0],Quality[▲ 1.0],Delivery[▼ 1.0],Tier[▲ 1.0]
S1,420.0,88.000000,12.000000,4.0
S2,380.0,91.000000,9.000000,5.0
S3,610.0,86.203523,20.000000,2.0
S4,350.0,75.000000,7.352990,3.0
S6,500.0,82.000000,14.000000,4.0
S7,460.0,86.200890,10.000000,5.0
S8,395.0,95.000000,9.471168,3.0


,Price[▼ 1.0],Quality[▲ 1.0],Delivery[▼ 1.0],Tier[▲ 1.0]
S1,420.0,88.0,12.0,4.0
S2,380.0,91.0,9.0,5.0
S3,610.0,85.0,20.0,2.0
S4,350.0,75.0,10.5,3.0
S6,500.0,82.0,14.0,4.0
S7,460.0,85.0,10.0,5.0
S8,395.0,95.0,10.5,3.0


In [6]:
import pandas as pd

comparison = pd.DataFrame(
    {
        "mean": mean_imputed.matrix["Quality"],
        "median": median_imputed.matrix["Quality"],
        "iterative": iterative_imputed.matrix["Quality"],
        "knn": knn_imputed.matrix["Quality"],
    }
)
comparison.loc[["S3", "S7"]]

,mean,median,iterative,knn
Alternatives,,,,
S3,86.2,88.0,86.203523,85.0
S7,86.2,88.0,86.200890,85.0


For `S3` the four strategies disagree quite a bit: `mean`/`median` only
see the *Quality* column (around 82-85), while `IterativeImputer`
notices that `S3` also has a high `Price` and predicts an even higher
`Quality` (~98), and `KNNImputer` looks for the two most similar
suppliers overall and lands in between. There is no universally "correct"
choice — `IterativeImputer`/`KNNImputer` are usually preferred when
criteria are correlated (as *Price* and *Quality* often are), while
`SimpleImputer` is a fast, transparent baseline.

## 5. Combined case: filter + impute + rank

Let's put everything together in a single `pipeline`
([introduced in the Quick Start](quickstart.ipynb)): drop the
blacklisted *Tier 1* supplier, impute the remaining missing values with
`KNNImputer`, and rank the survivors with
[TOPSIS](https://en.wikipedia.org/wiki/TOPSIS).

In [7]:
from skcriteria.preprocessing import invert_objectives, scalers
from skcriteria.agg.topsis import TOPSIS
from skcriteria.pipelines import mkpipe

pipe = mkpipe(
    filters.FilterNotIn({"Tier": [1]}),
    impute.KNNImputer(n_neighbors=2),
    invert_objectives.NegateMinimize(),
    scalers.VectorScaler(target="matrix"),
    TOPSIS(),
)
pipe

<SKCPipeline [steps=[('filternotin', <FilterNotIn [criteria_filters={'Tier': array([1])}, ignore_missing_criteria=False]>), ('knnimputer', <KNNImputer [keep_empty_criteria=False, metric='nan_euclidean', missing_values=nan, n_neighbors=2, weights='uniform']>), ('negateminimize', <NegateMinimize []>), ('vectorscaler', <VectorScaler [target='matrix']>), ('topsis', <TOPSIS [metric='euclidean']>)]]>

In [8]:
pipe.evaluate(dm)

Alternatives,S1,S2,S3,S4,S6,S7,S8
Rank,3,1,7,5,6,2,4


Starting from eight suppliers with a blacklisted tier and four missing
values, the pipeline discards `S5`, fills in the missing `Quality` and
`Delivery` figures using their nearest neighbors, and produces a
complete ranking of the remaining seven — all in one call, and in a way
that is trivial to repeat with a different imputer or a stricter filter.